In [84]:
import lightgbm as lgb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, root_mean_squared_error, r2_score

from sklearn.datasets import load_breast_cancer, fetch_california_housing

import joblib

### Binary Classification

In [61]:
data = load_breast_cancer()

In [62]:
features = pd.DataFrame(data.data, columns=data.feature_names)

In [63]:
target = pd.Series(data.target, name='target')

#There's some sort of class inbalance 60/40
pd.DataFrame({
    'Count': target.value_counts(),
    'Percentage (%)': (target.value_counts(normalize=True) * 100).round(2)
})

,Count,Percentage (%)
target,,
1,357,62.74
0,212,37.26


In [64]:
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2, random_state=42)

In [65]:
train_data = lgb.Dataset(features_train, label=target_train, params={"max_bin": 16})
test_data = lgb.Dataset(features_test, label=target_test, reference=train_data)

In [66]:
params = {
    #binary because it is a binary classification problem
    'objective': 'binary',
    'metric': 'binary_error',
    'boosting_type': 'gbdt',
    'learning_rate': 0.1,
    'verbose': -1,
#    'num_leaves': 31,
   'feature_fraction': 0.9 #adding this parameter to avoid overfitting, increased accuracy by 2%
}

In [67]:
model_classification = lgb.train(
    params, 
    train_data, 
    num_boost_round=100, 
    valid_sets=[test_data], 
    callbacks=[lgb.early_stopping(stopping_rounds=10)]
)

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[13]	valid_0's binary_error: 0.0175439


c:\Users\USER\Documents\light_gbm\.venv\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


In [68]:
train_pred = model_classification.predict(features_train, num_iteration=model_classification.best_iteration)
test_pred = model_classification.predict(features_test, num_iteration=model_classification.best_iteration)

In [69]:
train_pred_binary = np.round(train_pred)
test_pred_binary = np.round(test_pred)

In [ ]:
train_accuracy = accuracy_score(target_train, train_pred_binary)
test_accuracy = accuracy_score(target_test, test_pred_binary)

In [71]:
train_accuracy, test_accuracy

(0.9758241758241758, 0.9824561403508771)

In [72]:
from sklearn.metrics import classification_report
print(classification_report(target_test, test_pred_binary))   

              precision    recall  f1-score   support

           0       1.00      0.95      0.98        43
           1       0.97      1.00      0.99        71

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [73]:
cv_result = lgb.cv(
    {**params, "max_bin": 16},
    lgb.Dataset(features_train, label=target_train),
    num_boost_round=100,
    nfold=10,
    callbacks=[lgb.early_stopping(stopping_rounds=10)]
    
)   

#Done for cross validation, now we can use the best number of iterations to train the model on the entire training set and evaluate it on the test set. (optional)
#Mean error is around 2.8%, ~97-98% accuracy, meaning 16 bins was the right choice for the dataset

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[43]	valid's binary_error: 0.028599 + 0.0197782


### Regression

In [392]:
housing_data = fetch_california_housing()

housing_features = pd.DataFrame(housing_data.data, columns=housing_data.feature_names)
housing_target = pd.Series(housing_data.target, name='target')

housing_features

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25
...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32


In [393]:
features_train, features_test, target_train, target_test = train_test_split(housing_features, housing_target, test_size=0.2, random_state=42)

In [394]:
train_data = lgb.Dataset(features_train, label=target_train, 
                        # params={'max_bin': 128} change was minuscule, no need to change it
                        )
test_data = lgb.Dataset(features_test, label=target_test, reference=train_data)

In [415]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 30,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    #increasing min_data_in leaf reduced overfitting a bit, increasing accuracy by 1%
    # 'min_data_in_leaf': 10,
    # 'feature_pre_filter': False,
}

In [416]:
params['min_data_in_leaf'] = 20  # was 5; with 20k samples you can afford this   

In [417]:
model_regression = lgb.train(
    params,
    train_data, 
    num_boost_round=1000, 
    valid_sets=[test_data], 
    callbacks=[lgb.early_stopping(stopping_rounds=10)],
)

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[580]	valid_0's rmse: 0.436847


In [418]:
train_pred = model_regression.predict(features_train, num_iteration=model_regression.best_iteration)
test_pred = model_regression.predict(features_test, num_iteration=model_regression.best_iteration)

In [419]:
root_mean_squared_error(target_train, train_pred), root_mean_squared_error(target_test, test_pred) 


#Early stopping shows the RMSE, this is just for documentation purposes


(0.31038340538362835, 0.4368466746476338)

In [420]:
r2_score(target_train, train_pred), r2_score(target_test, test_pred)

(0.927932803296832, 0.8543698796944496)

In [421]:
joblib.dump(model_classification, 'lgb_model_classification.pkl')

['lgb_model_classification.pkl']

In [391]:
joblib.dump(model_regression, 'lgb_model_regression.pkl')

['lgb_model_regression.pkl']